# V91 - Chat-count sample weighting

Pre-registered plan: verbose users give more evidence, so weight training samples by log chat count. One fixed scheme, no sweep. Same word-only features and grouped folds under a fresh seed, same locks.

In [1]:
import re
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import Ridge
from sklearn.model_selection import StratifiedGroupKFold

SEED = 1616
N_SPLITS = 5
GATE_FLOOR = 0.002
rng = np.random.RandomState(SEED)

CANDIDATES = [Path("/kaggle/input/intelligo-data/data"), Path("data")]
DATA = next((p for p in CANDIDATES if (p / "train_relevance.csv").exists()), None)
assert DATA is not None, "train_relevance.csv not found"
OUT = Path("/kaggle/working" if Path("/kaggle/working").exists() else "results/v91")
OUT.mkdir(parents=True, exist_ok=True)
print("DATA:", DATA.resolve())

DATA: /home/athilla/Documents/IF_ITB/Lomba/IT-TODAY/intelligo-course-recommender/data


## Weighted vs unweighted Ridge, same folds

Weights are log1p chat counts normalized to mean one. Evaluation is unweighted user means as always.

In [2]:
MODS = [f"M_{i:03d}" for i in range(1, 18)]
train = pd.read_csv(DATA / "train_relevance.csv")
test_ids = pd.read_csv(DATA / "test.csv")["user_id"].tolist()
chats = pd.read_csv(DATA / "chat_history.csv")
assert not (set(train["user_id"]) & set(test_ids))
Y = train[MODS].to_numpy(float)
n_train = len(train)
by_user = chats.groupby("user_id")["user_chat_text"].apply(list)
docs = [" ".join(str(t) for t in by_user.get(u, [])) for u in list(train["user_id"])]
W = TfidfVectorizer(max_features=5000, ngram_range=(1, 1), min_df=3).fit_transform(docs).toarray()
sw = np.log1p(np.array([len(by_user.get(u, [])) for u in list(train["user_id"])]))
sw = sw / sw.mean()
print(f"weight range: {sw.min():.2f}-{sw.max():.2f}")

def template_group(texts):
    if not texts:
        return "NO_CHAT"
    return re.sub(r"\d+", "#", " ".join(str(texts[0]).lower().split()[:6]))
fold_groups = np.array([template_group(by_user.get(u, [])) for u in train["user_id"]])
strata = Y.argmax(1)

def ndcg5(yt, ys):
    order = np.argsort(-ys, axis=1)
    rel = np.take_along_axis(yt, order, 1)[:, :5]
    disc = 1.0 / np.log2(np.arange(2, 7))
    dcg = (rel * disc).sum(1)
    ideal = np.sort(yt, axis=1)[:, ::-1][:, :5]
    idcg = (ideal * disc).sum(1)
    return np.where(idcg > 0, dcg / np.maximum(idcg, 1e-12), 0.0)

def ridge_oof(weighted):
    oof = np.zeros_like(Y)
    fids = np.zeros(n_train, int)
    skf = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
    for fi, (tr, va) in enumerate(skf.split(W, strata, fold_groups)):
        for j in range(len(MODS)):
            m = Ridge(alpha=1.0)
            m.fit(W[tr], Y[tr, j], sample_weight=(sw[tr] if weighted else None))
            oof[va, j] = m.predict(W[va])
        fids[va] = fi + 1
    return oof, fids

oof_u, fid = ridge_oof(False)
oof_v, _ = ridge_oof(True)
base = ndcg5(Y, oof_u)
d = ndcg5(Y, oof_v) - base
lo, hi = np.quantile([rng.choice(d, len(d), replace=True).mean() for _ in range(2000)], [0.025, 0.975])
wf = min(ndcg5(Y[fid == k], oof_v[fid == k]).mean() - base[fid == k].mean() for k in range(1, N_SPLITS + 1))
print(f"unweighted={base.mean():.5f} weighted={ndcg5(Y, oof_v).mean():.5f} delta={d.mean():+.5f} CI95=[{lo:+.5f},{hi:+.5f}] worst-fold={wf:+.5f}")
print("ADOPT weighting" if (lo > GATE_FLOOR and wf >= 0) else "KEEP unweighted; verbosity weighting adds nothing")

weight range: 0.00-1.98


unweighted=0.62713 weighted=0.62509 delta=-0.00204 CI95=[-0.00388,-0.00024] worst-fold=-0.00612
KEEP unweighted; verbosity weighting adds nothing
